<a href="https://colab.research.google.com/github/marvinjc/fund_redes_neuronales_python/blob/main/laboratorio_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SINT-666 · Laboratorio 2 — Empresa de delivery
**Caso ficticio: RutaGo Delivery**

Archivo: `datos_lab2_delivery.csv`. Debes identificar y resolver dos necesidades: **entrega tardía** y **tiempo de entrega en minutos**. Además, clasifica conceptualmente el objetivo opcional **combustible consumido en litros**.

No se entregan arquitecturas resueltas. Justifica cada decisión y entrega este notebook con tus respuestas.

**Importante:** todos los datos son sintéticos; no se deben utilizar para decisiones reales.

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

keras.utils.set_random_seed(42)
print('TensorFlow:', tf.__version__)

# El CSV se consume directamente desde GitHub (no hace falta subir archivos ni montar Drive)
URL_DATOS = "https://raw.githubusercontent.com/marvinjc/fund_redes_neuronales_python/main/datos_lab2_delivery.csv"
df = pd.read_csv(URL_DATOS)
display(df.head())

TensorFlow: 2.21.0


,distancia_km,peso_carga_kg,lluvia,trafico_nivel,experiencia_repartidor_anios,entrega_tardia,tiempo_entrega_min,combustible_litros
0,27.39,4.09,0,0,6.97,1,79.35,1.78
1,25.07,6.60,0,2,3.76,1,99.98,1.60
2,4.47,1.34,0,1,2.01,0,29.23,0.25
3,2.55,6.45,0,2,4.88,1,45.12,0.27
4,22.94,7.28,1,0,3.87,0,77.19,1.81


## 1. Análisis del escenario

**1. ¿Qué variables están disponibles *antes* de realizar la entrega?**

Las cinco entradas: `distancia_km`, `peso_carga_kg`, `lluvia`, `trafico_nivel` y `experiencia_repartidor_anios`. Se conocen al planificar la ruta.

**2. ¿Cuál objetivo es clasificación y cuál regresión? ¿Por qué?**

- `entrega_tardia` (0/1) es **clasificación binaria**: la respuesta es una de dos clases.
- `tiempo_entrega_min` es **regresión**: la respuesta es un número continuo en minutos.
- (Reto adicional) `combustible_litros` es un valor continuo, por lo tanto **regresión**.

**3. ¿Qué activación de salida y qué función de pérdida elegirías en cada caso?**

| | Modelo A: entrega tardía | Modelo B: tiempo (min) |
|---|---|---|
| Salida | `Dense(1, sigmoid)` → probabilidad entre 0 y 1 | `Dense(1, linear)` → valor continuo |
| Pérdida | `binary_crossentropy` | `mse` |
| Métrica | `accuracy` | `mae` (minutos) |

**4. ¿Por qué no debemos usar columnas objetivo como entradas?**

`entrega_tardia`, `tiempo_entrega_min` y `combustible_litros` son **resultados que solo se conocen después de la entrega**. Usarlas como entrada produce *fuga de información* (*data leakage*): el modelo parecería excelente en el laboratorio, pero en la práctica no tendría esos datos al momento de predecir.

Como explicó el profesor, sería como hacer trampa en un examen: el modelo ya conocería las respuestas antes de predecir.

In [2]:
# Revisión rápida de calidad de datos
print('Dimensiones:', df.shape)
print('Valores faltantes:', df.isna().sum().sum())
display(df.describe().T)

Dimensiones: (2000, 8)
Valores faltantes: 0


,count,mean,std,min,25%,50%,75%,max
distancia_km,2000.0,14.442860,7.796172,0.81,7.6400,14.475,21.3400,27.99
peso_carga_kg,2000.0,9.136555,5.164916,0.20,4.4200,9.400,13.6000,17.99
lluvia,2000.0,0.323000,0.467740,0.00,0.0000,0.000,1.0000,1.00
trafico_nivel,2000.0,0.946000,0.742536,0.00,0.0000,1.000,1.2500,2.00
experiencia_repartidor_anios,2000.0,4.154350,2.251511,0.20,2.1475,4.220,6.0825,8.00
entrega_tardia,2000.0,0.538000,0.498679,0.00,0.0000,1.000,1.0000,1.00
tiempo_entrega_min,2000.0,66.132770,22.864901,8.00,49.0350,65.050,82.2450,134.25
combustible_litros,2000.0,1.120930,0.527972,0.20,0.6800,1.110,1.5700,2.32


In [3]:
# 1) Selección de las 5 entradas y los dos objetivos (combustible_litros queda fuera: es el reto conceptual)
columnas_entrada = ['distancia_km', 'peso_carga_kg', 'lluvia', 'trafico_nivel', 'experiencia_repartidor_anios']
X = df[columnas_entrada]
y_tardia = df['entrega_tardia']
y_tiempo = df['tiempo_entrega_min']

print('Entradas:', X.shape)
print('Objetivo entrega tardía:', y_tardia.shape)
print('Objetivo tiempo de entrega:', y_tiempo.shape)
print('Proporción de entregas tardías:', round(y_tardia.mean(), 3))

Entradas: (2000, 5)
Objetivo entrega tardía: (2000,)
Objetivo tiempo de entrega: (2000,)
Proporción de entregas tardías: 0.538


In [4]:
# 2) Una sola partición (80/20) con los MISMOS índices para ambos objetivos
#    stratify=y_tardia conserva la proporción de entregas tardías en train y test
indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, stratify=y_tardia, random_state=42)

X_train = X.iloc[idx_train]
X_test = X.iloc[idx_test]
y_tardia_train = y_tardia.iloc[idx_train]
y_tardia_test = y_tardia.iloc[idx_test]
y_tiempo_train = y_tiempo.iloc[idx_train]
y_tiempo_test = y_tiempo.iloc[idx_test]

print('Entrena', len(idx_train), 'pruebas', len(idx_test))
assert len(idx_train) + len(idx_test) == len(X)

Entrena 1600 pruebas 400


In [5]:
# 3) StandardScaler: fit SOLO con entrenamiento (z = (x - media) / desviación)
#    El mismo escalador de entradas se comparte entre las dos redes
escalado = StandardScaler()
X_train_s = escalado.fit_transform(X_train)   # aprende media y desviación y transforma train
X_test_s = escalado.transform(X_test)         # reutiliza los parámetros de train sobre test

print('Primer registro original:', X_train.iloc[0].to_dict())
print('Primer registro escalado:', np.round(X_train_s[0], 3))
assert X_train_s.shape[1] == 5

Primer registro original: {'distancia_km': 24.57, 'peso_carga_kg': 15.25, 'lluvia': 1.0, 'trafico_nivel': 2.0, 'experiencia_repartidor_anios': 3.38}
Primer registro escalado: [ 1.302  1.189  1.483  1.392 -0.351]


## 2. Modelo A

Explica tu arquitectura, activación de salida, optimizador, pérdida y métrica antes de escribir el código.

**Modelo A — clasificación binaria: ¿la entrega llegará tarde?  Arquitectura 5 → 12 → 6 → 1**

- **Entrada `Input(shape=(5,))`:** cinco variables escaladas por registro.
- **Dos capas ocultas `Dense` con ReLU (12 y 6 neuronas):** el problema es pequeño (5 entradas, 2 000 registros), por lo que no requiere una red grande. Dos capas permiten combinar las variables de forma no lineal (por ejemplo, lluvia + tráfico alto + distancia larga) y pasar de 12 a 6 neuronas comprime la información hacia la decisión final. ReLU es una activación sencilla y estándar para capas ocultas.
- **Salida `Dense(1, sigmoid)`:** produce un valor entre 0 y 1 que se interpreta como la **probabilidad estimada de entrega tardía**.
- **Optimizador `Adam` (lr = 0.001):** actualiza los pesos con gradientes y adapta el paso por parámetro; es una elección inicial frecuente.
- **Pérdida `binary_crossentropy`:** penaliza la diferencia entre la etiqueta 0/1 y la probabilidad predicha; es la pérdida coherente con Sigmoid.
- **Métrica `accuracy`:** proporción de aciertos con umbral 0.5. Las clases están razonablemente balanceadas (~54 % tardías), por lo que es una primera aproximación válida.

**Compilar no entrena:** `compile()` define cómo se actualizan los pesos y cómo se mide el error. La **pérdida** es lo que el optimizador minimiza; la **métrica** es lo que usamos para interpretar el rendimiento. `fit()` es lo que sí ajusta los pesos.

In [6]:
# Construcción
modelo_tardia = keras.Sequential([
    layers.Input(shape=(5,)),
    layers.Dense(12, activation='relu'),
    layers.Dense(6, activation='relu'),
    layers.Dense(1, activation='sigmoid')
], name='clasificador_entrega_tardia')

modelo_tardia.summary()

Model: "clasificador_entrega_tardia"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 12)             │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │            78 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             7 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 157 (628.00 B)

 Trainable params: 157 (628.00 B)

 Non-trainable params: 0 (0.00 B)

In [7]:
# Compilación
modelo_tardia.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
print('Clasificador compilado')

Clasificador compilado


In [8]:
# Entrenamiento breve (no se usa el conjunto de prueba para entrenar)
hist_tardia = modelo_tardia.fit(X_train_s, y_tardia_train, epochs=18, batch_size=32)

print('Pérdida final de entrenamiento:', round(hist_tardia.history['loss'][-1], 4))
print('Accuracy final de entrenamiento:', round(hist_tardia.history['accuracy'][-1], 4))

Epoch 1/18
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.4238 - loss: 0.7485
Epoch 2/18
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5294 - loss: 0.6958
Epoch 3/18
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5825 - loss: 0.6672
Epoch 4/18
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6012 - loss: 0.6461
Epoch 5/18
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6269 - loss: 0.6282
Epoch 6/18
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6500 - loss: 0.6112
Epoch 7/18
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6737 - loss: 0.5952
Epoch 8/18
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.6881 - loss: 0.5809
Epoch 9/18
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.7013 - loss: 0.5693
Epoch 10/18
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7081 - loss: 0.5612
Epoch 11/18
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7144 - loss: 0.5559
Epoch 12/18
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7113 - l

In [9]:
# Evaluación y predicciones (umbral ilustrativo de 0.50)
loss_t, acc_t = modelo_tardia.evaluate(X_test_s, y_tardia_test, verbose=0)
prob_tardia = modelo_tardia.predict(X_test_s[:5], verbose=0).ravel()
pred_tardia = (prob_tardia >= 0.5).astype(int)

print(f'Prueba | loss={loss_t:.4f} | accuracy={acc_t:.3f}')
display(pd.DataFrame({'probabilidad_tardia': np.round(prob_tardia, 3),
                      'clase_predicha': pred_tardia,
                      'clase_real': y_tardia_test.iloc[:5].astype(int).values}))

Prueba | loss=0.5856 | accuracy=0.695


,probabilidad_tardia,clase_predicha,clase_real
0,0.688,1,1
1,0.773,1,0
2,0.852,1,1
3,0.301,0,1
4,0.440,0,1


## 3. Modelo B

Explica las diferencias respecto al modelo A.

**Modelo B — regresión: ¿cuántos minutos tardará la entrega?** Se conservan las mismas entradas, el mismo escalador y las mismas capas ocultas (**5 → 12 → 6 → 1**). Cambian el objetivo y la configuración de salida, pérdida y métrica:

| Decisión | Modelo A: entrega tardía | Modelo B: tiempo (min) |
|---|---|---|
| Tipo de tarea | Clasificación binaria | Regresión |
| Entradas | 5 variables escaladas | Las mismas 5 variables escaladas |
| Capas ocultas | `Dense` 12 + 6, ReLU | `Dense` 12 + 6, ReLU |
| Última capa | `Dense(1, sigmoid)` | `Dense(1, linear)` |
| Optimizador | Adam | Adam |
| Pérdida | `binary_crossentropy` | `mse` |
| Métrica | `accuracy` | `mae` (minutos) |
| Salida | Probabilidad (0–1) | Minutos estimados |

- **Salida `linear`:** devuelve un valor continuo sin límites; los tiempos llegan hasta ~134 minutos, algo que Sigmoid no podría representar.
- **`mse`:** promedio de los errores al cuadrado; penaliza especialmente los errores grandes.
- **`mae`:** promedio de diferencias absolutas; se interpreta directamente en **minutos**.
- No se usa `accuracy`: no hay clases discretas que acertar.
- El objetivo se mantiene en minutos para que el MAE sea fácil de interpretar. Como el valor medio (~66 min) es grande frente a la salida inicial de la red, se usan más épocas que en el Modelo A para que converja.

In [10]:
# Construcción
modelo_tiempo = keras.Sequential([
    layers.Input(shape=(5,)),
    layers.Dense(12, activation='relu'),
    layers.Dense(6, activation='relu'),
    layers.Dense(1, activation='linear')
], name='regresor_tiempo_entrega')

modelo_tiempo.summary()

Model: "regresor_tiempo_entrega"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 12)             │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 6)              │            78 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │             7 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 157 (628.00 B)

 Trainable params: 157 (628.00 B)

 Non-trainable params: 0 (0.00 B)

In [11]:
# Compilación
modelo_tiempo.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)
print('Regresor compilado')

Regresor compilado


In [12]:
# Entrenamiento breve
hist_tiempo = modelo_tiempo.fit(X_train_s, y_tiempo_train, epochs=100, batch_size=32, verbose=0)

print('MSE final de entrenamiento:', round(hist_tiempo.history['loss'][-1], 2))
print('MAE final de entrenamiento:', round(hist_tiempo.history['mae'][-1], 2), 'min')

MSE final de entrenamiento: 29.74
MAE final de entrenamiento: 4.33 min


In [13]:
# Evaluación y predicciones
loss_m, mae_m = modelo_tiempo.evaluate(X_test_s, y_tiempo_test, verbose=0)
pred_min = modelo_tiempo.predict(X_test_s[:5], verbose=0).ravel()

print(f'Prueba | MSE={loss_m:.2f} | MAE={mae_m:.2f} min')
display(pd.DataFrame({'minutos_predichos': np.round(pred_min, 2),
                      'minutos_reales': np.round(y_tiempo_test.iloc[:5].values, 2)}))

Prueba | MSE=30.63 | MAE=4.36 min


,minutos_predichos,minutos_reales
0,88.690002,80.59
1,88.900002,83.89
2,92.059998,90.09
3,60.820000,61.82
4,48.220001,41.24


## 4. Cierre

1. Presenta una predicción de cada modelo, con unidades y significado.
2. Justifica por qué los dos modelos no tienen la misma activación de salida.
3. Indica qué harías con el objetivo opcional `combustible_litros` (sin entrenar una tercera red).
4. Explica una limitación de trabajar con datos sintéticos.

### 4.1 Predicción de una entrega nueva

Se usa `transform` (nunca `fit_transform`) con el escalador ya ajustado, y el mismo orden de columnas del entrenamiento. **Entrega de ejemplo:** 18 km, 9 kg de carga, con lluvia, tráfico alto (2) y repartidor con 2 años de experiencia.

In [14]:
nueva_entrega = pd.DataFrame([{
    'distancia_km': 18.0,
    'peso_carga_kg': 9.0,
    'lluvia': 1,
    'trafico_nivel': 2,
    'experiencia_repartidor_anios': 2.0
}], columns=columnas_entrada)

nueva_s = escalado.transform(nueva_entrega).astype('float32')
p_tardia = float(modelo_tardia.predict(nueva_s, verbose=0)[0, 0])
minutos = float(modelo_tiempo.predict(nueva_s, verbose=0)[0, 0])

print(f'Modelo A | Probabilidad estimada de entrega tardía: {p_tardia:.1%}')
print(f'Modelo A | Clase predicha (umbral 0.5): {"TARDÍA" if p_tardia >= 0.5 else "A TIEMPO"}')
print(f'Modelo B | Tiempo de entrega estimado: {minutos:.1f} minutos')

Modelo A | Probabilidad estimada de entrega tardía: 93.6%
Modelo A | Clase predicha (umbral 0.5): TARDÍA
Modelo B | Tiempo de entrega estimado: 94.3 minutos


In [15]:
# Verificaciones básicas (no sustituyen una evaluación rigurosa)
assert not df.isna().any().any(), 'Hay valores faltantes'
assert set(df['entrega_tardia'].unique()).issubset({0, 1})
assert (df['tiempo_entrega_min'] > 0).all()
assert modelo_tardia.input_shape[-1] == modelo_tiempo.input_shape[-1] == 5
assert np.all((prob_tardia >= 0) & (prob_tardia <= 1))
assert np.isfinite(pred_min).all()
assert np.isfinite([loss_t, acc_t, loss_m, mae_m]).all()
print('Verificaciones básicas superadas.')

Verificaciones básicas superadas.


### Respuestas del cierre

**1. Predicción de cada modelo, con unidades y significado.**
- **Modelo A:** la salida es una **probabilidad** entre 0 y 1 (sin unidades). Por ejemplo, 0.85 significa que el modelo estima 85 % de probabilidad de que esa entrega llegue tarde; con umbral 0.5 se clasifica como "tardía".
- **Modelo B:** la salida es un **tiempo en minutos**. Por ejemplo, 95 significa que el modelo estima que esa entrega tardará unos 95 minutos; el **MAE** indica cuántos minutos se equivoca en promedio.
- Los valores concretos de la entrega de ejemplo se muestran en la celda anterior.

**2. ¿Por qué los dos modelos no tienen la misma activación de salida?**
Porque son dos tipos de problema distintos: el Modelo A es un problema de **clasificación** (binaria) y el Modelo B es un problema de **regresión**. En A (clasificación: ¿llega tarde o no?) se necesita una **probabilidad acotada entre 0 y 1** (Sigmoid) emparejada con `binary_crossentropy`. En B (regresión: ¿cuántos minutos tardará?) se necesita un **número real en minutos**, sin límite superior fijo (Linear), emparejado con `mse`. La capa de salida debe reflejar la naturaleza del objetivo.

**3. ¿Qué haría con `combustible_litros`?**
Es **regresión** (cantidad continua y no negativa, p. ej. 0.25 o 1.78 L). Usaría las mismas cinco entradas escaladas con el mismo `StandardScaler`, salida `Dense(1, linear)` (o `relu`/`softplus` para evitar valores negativos), pérdida `mse` (o `huber` si hubiera atípicos) y métrica `mae` en litros. No usaría `entrega_tardia` ni `tiempo_entrega_min` como entradas, porque son resultados posteriores a la entrega (fuga de información).

**4. Limitación de trabajar con datos sintéticos.**
Los datos se generan con reglas simples, por lo que **no reflejan la complejidad, el ruido ni los sesgos de una operación real** (accidentes, zonas, clima extremo, comportamiento de clientes). El modelo puede parecer preciso, pero no hay garantía de que generalice a datos reales; no debe usarse para decisiones reales.

### Preguntas de análisis del enunciado

**¿Por qué dos problemas de una misma empresa pueden requerir configuraciones distintas de salida?**
Porque el tipo de problema cambia: "¿llegará tarde?" es un problema de **clasificación**, una decisión de dos clases (probabilidad con Sigmoid), y "¿cuántos minutos?" es un problema de **regresión**, una cantidad continua (salida lineal). Aunque compartan entradas, la capa de salida, la pérdida y la métrica deben ajustarse al objetivo.

**¿Qué sucedería si se utilizara Sigmoid para predecir minutos directamente?**
La salida quedaría limitada entre 0 y 1 y no podría representar tiempos como 45 o 100 minutos. Además, la activación se saturaría intentando acercarse a valores grandes y el aprendizaje sería deficiente (gradientes casi nulos).

**¿Cuál es la diferencia entre función de pérdida y métrica?**
La **pérdida** es la función que el optimizador minimiza durante el entrenamiento (debe ser diferenciable), por ejemplo `binary_crossentropy` o `mse`. La **métrica** es una medida para que las personas interpreten el rendimiento (`accuracy`, `mae` en minutos); no se usa para actualizar los pesos.

**¿Por qué el escalador debe ajustarse solo con los datos de entrenamiento?**
Para evitar **fuga de información**. Si media y desviación se calculan con todos los datos, el conjunto de prueba influye en el preprocesamiento y la evaluación resulta demasiado optimista. En producción los datos nuevos tampoco existen al entrenar, por eso se usa `transform` con los parámetros aprendidos en entrenamiento.

**¿Qué limitación tiene entrenar con datos ficticios?**
Los patrones son artificiales y más simples que los reales: el modelo aprende relaciones que pueden no existir en la operación real, y su desempeño no garantiza funcionamiento en producción.